<a href="https://colab.research.google.com/github/askarbekkk/kyrgyz-embeddings/blob/main/kyrgyz-embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install datasets -q
from datasets import load_dataset
import random

ds = load_dataset("Zhantas/Cleaned-Kyrgyz_Wikipedia", split="train")

chunks = []

for art in ds.select(range(5000)):
  for para in art["text"].split("\n"):
    if 200 < len(para) < 800:
      chunks.append({"title": art["title"], "text": para.strip()})

print(len(chunks))
random.seed(42)

for c in random.sample(chunks, 3):
  print(c["title"], "|", c["text"][:200], "\n")



README.md:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 67.2MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/76519 [00:00<?, ? examples/s]

14250
Саймалыташ сүрөт галереясы | Саймалыташ сүрөт галереясын 1902-жылы орус армиясынын офицери, топограф Н.Г.Хлудов ачкан. Кийин аны генерал –мойор И.Т. Пославский (1902-1903), Б.М.Зима (1946), А.Н.Бернштам(1950) изилдеген. Саймалыта 

Кыргыз тилинин грамматикасы | Баяндагыч ыңгай: Кыймыл-аракеттин ошол учурда болуп өткөнүн, болуп жатканын, боло турганын жайынча баяндаган этиш сөздөр баяндагыч ыңгай деп аталат. Мисалы: Шаарга бардым. Шаарга бара жатам. Шаарга ба 

Айыл чарба | Асыл тукум мал чарбасын 228 чарба жүргүзүүчү субъект түзөт. Айыл чарбаны пландоо жана кеңеш берүү жактан камсыз кылуу үчүн агрардык илим жана консулътациялык кызмат борбору, анын карамагындагы төрт ил 



In [11]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
from google.colab import userdata
api_key = userdata.get('GEMINI_API_KEY')


In [13]:
!pip install google-genai -q

from google import genai
from google.colab import userdata, drive
import random, time, json

drive.mount('/content/drive')
SAVE_PATH = "/content/drive/MyDrive/pairs_raw.json"

client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))

PROMPT = """You are given a passage in the Kyrgyz language.

Write ONE short question in Kyrgyz that can be answered using this passage.
The question must sound like something a real person would type into a search engine — natural, specific, and self-contained.

Rules:
- Write the question in Kyrgyz only.
- Do not copy full sentences from the passage.
- Do not reference "the text" or "the passage".
- Return only the question, with no explanation or extra formatting.

Passage:
{text}"""

random.seed(42)
sample = random.sample(chunks, 300)
MODELS = ["gemini-3.5-flash-lite", "gemini-3.1-flash-lite", "gemini-3.6-flash"]

def save():
    with open(SAVE_PATH, "w", encoding="utf-8") as f:
        json.dump(pairs, f, ensure_ascii=False, indent=2)

pairs = []
for i, c in enumerate(sample):
    for attempt in range(3):
        try:
            r = client.models.generate_content(
                model=MODELS[attempt % len(MODELS)],
                contents=PROMPT.format(text=c["text"]),
            )
            pairs.append({
                "query": r.text.strip(),
                "positive": c["text"],
                "title": c["title"],
            })
            break
        except Exception as e:
            print(f"[{i}] attempt {attempt+1}: {e}")
            time.sleep(10 * (attempt + 1))

    if i % 25 == 0:
        save()
        print(f"processed {i}, collected {len(pairs)}, saved")
    time.sleep(4.2)

save()
print(f"\nDONE: {len(pairs)} pairs saved to {SAVE_PATH}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
processed 0, collected 1, saved
[9] attempt 1: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
[15] attempt 1: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
processed 25, collected 26, saved
[32] attempt 1: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
processed 50, collected 51, saved
processed 75, collected 76, saved
processed 100, collected 101, saved
processed 125, collected 126, saved
processed 150, collected 151, sav

In [14]:
c = sample[0]
r = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=PROMPT.format(text=c["text"]),
)
print("QUERY:", r.text.strip())
print("\nTEXT:", c["text"][:300])

QUERY: Саймалыташ галереясын ким ачкан?

TEXT: Саймалыташ сүрөт галереясын 1902-жылы орус армиясынын офицери, топограф Н.Г.Хлудов ачкан. Кийин аны генерал –мойор И.Т. Пославский (1902-1903), Б.М.Зима (1946), А.Н.Бернштам(1950) изилдеген. Саймалыташтагы эстеликтерде артына куйрук сымал нерсеси бар адамдардын сөлөкөттөрү сакталган. Алардын көпчүлү


In [15]:
!ls -la /content/*.json
!ls -la /content/drive/MyDrive/*.json

ls: cannot access '/content/*.json': No such file or directory
-rw------- 1 root root 270554 Sep  8 10:35 /content/drive/MyDrive/pairs_raw.json


In [16]:
from google.colab import drive
drive.mount('/content/drive')

import json
with open("/content/drive/MyDrive/pairs_raw.json", "w", encoding="utf-8") as f:
    json.dump(pairs, f, ensure_ascii=False, indent=2)
print("saved", len(pairs))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
saved 300


In [ ]:
CHECK = """You are given a passage in Kyrgyz and a question in Kyrgyz.

Decide: can this question be fully answered using ONLY this passage?

Answer with exactly one word: YES or NO.

Passage:
{text}

Question:
{query}"""


In [23]:
!pip install groq -q
from groq import Groq
from google.colab import userdata

groq_client = Groq(api_key=userdata.get('groq_apikey'))


r = groq_client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[{"role": "user", "content": CHECK.format(text=pairs[0]["positive"], query=pairs[0]["query"])}],
)
print(r.choices[0].message.content)

YES


In [24]:
clean = []
for i, p in enumerate(pairs):
    for attempt in range(3):
        try:
            r = groq_client.chat.completions.create(
                model="openai/gpt-oss-120b",
                messages=[{"role": "user", "content": CHECK.format(text=p["positive"], query=p["query"])}],
            )
            if "YES" in r.choices[0].message.content.strip().upper():
                clean.append(p)
            break
        except Exception as e:
            print(f"[{i}] {e}")
            time.sleep(5 * (attempt + 1))

    if i % 25 == 0:
        print(f"{i}: kept {len(clean)}")
        with open("/content/drive/MyDrive/pairs_clean.json", "w", encoding="utf-8") as f:
            json.dump(clean, f, ensure_ascii=False, indent=2)
    time.sleep(1)

with open("/content/drive/MyDrive/pairs_clean.json", "w", encoding="utf-8") as f:
    json.dump(clean, f, ensure_ascii=False, indent=2)

print(f"\n{len(clean)} from {len(pairs)} passed the check")

0: kept 1
25: kept 25
50: kept 48
75: kept 69
100: kept 93
[122] Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01m20eaqqveaksy07f5jn6p7yp` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 7232, Requested 855. Please try again in 652.5ms. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
125: kept 117
[132] Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01m20eaqqveaksy07f5jn6p7yp` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 7239, Requested 902. Please try again in 1.057499999s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
150: kept 141
[162] Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
!pip install -U sentence-transformers -q

import json, random
from sentence_transformers import SentenceTransformer, losses, SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from datasets import Dataset

with open("/content/drive/MyDrive/pairs_clean.json", encoding="utf-8") as f:
    clean = json.load(f)

random.seed(42)
random.shuffle(clean)
split = int(len(clean) * 0.8)
train_pairs, test_pairs = clean[:split], clean[split:]
print(len(train_pairs), len(test_pairs))

# тестовый набор для оценки
corpus  = {str(i): p["positive"] for i, p in enumerate(test_pairs)}
queries = {str(i): p["query"]    for i, p in enumerate(test_pairs)}
relevant = {str(i): {str(i)} for i in range(len(test_pairs))}

evaluator = InformationRetrievalEvaluator(
    queries=queries, corpus=corpus, relevant_docs=relevant, name="ky-test"
)

model = SentenceTransformer("intfloat/multilingual-e5-small")

# ДО обучения
before = evaluator(model)
print("BEFORE:", before)

222 56


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BEFORE: {'ky-test_cosine_accuracy@1': 1.0, 'ky-test_cosine_accuracy@3': 1.0, 'ky-test_cosine_accuracy@5': 1.0, 'ky-test_cosine_accuracy@10': 1.0, 'ky-test_cosine_precision@1': 1.0, 'ky-test_cosine_precision@3': 0.3333333333333333, 'ky-test_cosine_precision@5': 0.19999999999999998, 'ky-test_cosine_precision@10': 0.09999999999999999, 'ky-test_cosine_recall@1': 1.0, 'ky-test_cosine_recall@3': 1.0, 'ky-test_cosine_recall@5': 1.0, 'ky-test_cosine_recall@10': 1.0, 'ky-test_cosine_ndcg@10': 1.0, 'ky-test_cosine_mrr@10': 1.0, 'ky-test_cosine_map@100': 1.0}


In [8]:
# запросы только из теста, а корпус — из всех чанков
corpus = {str(i): c["text"] for i, c in enumerate(chunks[:5000])}


queries, relevant = {}, {}
for qi, p in enumerate(test_pairs):
    doc_id = f"gold_{qi}"
    corpus[doc_id] = p["positive"]
    queries[str(qi)] = p["query"]
    relevant[str(qi)] = {doc_id}

print(len(corpus), len(queries))

evaluator = InformationRetrievalEvaluator(
    queries=queries, corpus=corpus, relevant_docs=relevant, name="ky-test"
)

before = evaluator(model)
print("BEFORE:", before)

5056 56
BEFORE: {'ky-test_cosine_accuracy@1': 0.5535714285714286, 'ky-test_cosine_accuracy@3': 0.9285714285714286, 'ky-test_cosine_accuracy@5': 0.9285714285714286, 'ky-test_cosine_accuracy@10': 0.9464285714285714, 'ky-test_cosine_precision@1': 0.5535714285714286, 'ky-test_cosine_precision@3': 0.3095238095238095, 'ky-test_cosine_precision@5': 0.1857142857142857, 'ky-test_cosine_precision@10': 0.09464285714285714, 'ky-test_cosine_recall@1': 0.5535714285714286, 'ky-test_cosine_recall@3': 0.9285714285714286, 'ky-test_cosine_recall@5': 0.9285714285714286, 'ky-test_cosine_recall@10': 0.9464285714285714, 'ky-test_cosine_ndcg@10': 0.7844323105442259, 'ky-test_cosine_mrr@10': 0.7287414965986395, 'ky-test_cosine_map@100': 0.7322030350601779}


In [9]:
train_ds = Dataset.from_dict({
    "anchor":   [p["query"]    for p in train_pairs],
    "positive": [p["positive"] for p in train_pairs],
})

loss = losses.MultipleNegativesRankingLoss(model)

args = SentenceTransformerTrainingArguments(
    output_dir="/content/ky-e5-base ",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    fp16=True,
    logging_steps=10,
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    loss=loss,
)
trainer.train()

after = evaluator(model)
print("BEFORE accuracy@1:", before['ky-test_cosine_accuracy@1'])
print("AFTER  accuracy@1:", after['ky-test_cosine_accuracy@1'])
print("BEFORE nDCG@10:", before['ky-test_cosine_ndcg@10'])
print("AFTER  nDCG@10:", after['ky-test_cosine_ndcg@10'])

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 0, 'pad_token_id': 1}.


Step,Training Loss
10,1.069468
20,0.184237
30,0.086774
40,0.031045


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

BEFORE accuracy@1: 0.5535714285714286
AFTER  accuracy@1: 0.5714285714285714
BEFORE nDCG@10: 0.7844323105442259
AFTER  nDCG@10: 0.8101972701297863
